# Seaborn Phase 6: Small Multiples (Deep Diving)
### Credit Card Risk Analysis Project

A single chart with `hue=` starts to fall apart once you have more than 2-3 groups —
too many colors overlapping becomes unreadable. Small multiples solve this by giving
each group its own tiny chart, all built with identical axes so they stay directly
comparable.

This notebook covers 2 topics:
11. **FacetGrid / catplot** — a grid of identical charts, split by one or two categories
12. **Pairplot** — automatically grid every numeric column against every other

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first — it adds an `Education_Level` column so we have a
faceting variable with more than two categories to split on.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid")

n = 600

age = np.clip(np.random.normal(40, 12, n), 18, 80)
education_level = np.random.choice(
    ['High School', 'Bachelor', 'Graduate'], size=n, p=[0.35, 0.45, 0.20]
)

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
# Higher education levels tend toward somewhat higher credit limits in this dataset
education_bump = pd.Series(education_level).map(
    {'High School': 1.0, 'Bachelor': 1.15, 'Graduate': 1.3}
).values
credit_limit = np.clip(3000 + annual_income * 0.15 * education_bump + np.random.normal(0, 1500, n), 500, None)

credit_score = np.clip(np.random.normal(660, 65, n), 300, 850)
total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = np.clip((total_debt / annual_income) * 100, 0, 90)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
default_probability = np.clip(raw_risk + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

df = pd.DataFrame({
    'Age': age,
    'Education_Level': education_level,
    'Annual_Income': annual_income,
    'Credit_Limit': credit_limit,
    'Credit_Score': credit_score,
    'Debt_to_Income': debt_to_income,
    'Default_Probability': default_probability,
    'Default': default
})

print(df.shape)
df.head()


---
## Section 11: FacetGrid / catplot

`FacetGrid` is Seaborn's general-purpose small-multiples engine: give it a categorical
column to split on, and it builds one panel per category, then lets you `.map()` any
plotting function onto every panel at once. `catplot` is a higher-level shortcut built
specifically for categorical charts (bar, box, violin, etc.) that handles most of the
same faceting with less setup.


**Q1.** Create `g = sns.FacetGrid(df, col='Education_Level')`, then call `g.map(sns.scatterplot, 'Age', 'Credit_Limit')` — this is exactly the phase's example: the same Age-vs-Credit-Limit scatterplot, generated once per Education Level automatically.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
g = sns.FacetGrid(df, col='Education_Level')
g.map(sns.scatterplot, 'Age', 'Credit_Limit')
plt.show()


**Q2.** Create a `FacetGrid` with both `row='Default'` and `col='Education_Level'` — a 2x3 grid of panels, one for every combination of default status and education level. Map the same Age vs. Credit Limit scatterplot onto it.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
g = sns.FacetGrid(df, row='Default', col='Education_Level')
g.map(sns.scatterplot, 'Age', 'Credit_Limit')
plt.show()


**Q3.** Create `g = sns.FacetGrid(df, col='Education_Level', hue='Default', palette={0: 'steelblue', 1: 'tomato'})`, map the scatterplot as before, and call `g.add_legend()` afterward — combining faceting (columns) with color-coding (hue) within each panel.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
g = sns.FacetGrid(df, col='Education_Level', hue='Default', palette={0: 'steelblue', 1: 'tomato'})
g.map(sns.scatterplot, 'Age', 'Credit_Limit')
g.add_legend()
plt.show()


**Q4.** Suppose you were faceting on a column with 6+ categories instead of 3 — all in one row would get cramped. Simulate this by creating a `FacetGrid` on `Education_Level` with `col_wrap=2` (wraps panels onto a new row after every 2), even though we only have 3 categories here, just to see the wrapping behavior.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
g = sns.FacetGrid(df, col='Education_Level', col_wrap=2, height=4)
g.map(sns.histplot, 'Credit_Score')
plt.show()


**Q5.** Use the higher-level `sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')` — same faceting idea as `FacetGrid`, but `catplot` handles the categorical plot type directly through the `kind=` argument, with no separate `.map()` call needed.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')
plt.show()


**Q6.** Repeat Q5 with `kind='bar'` instead of `'box'` — one bar chart of average `Debt_to_Income` by `Default`, per Education Level panel.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='bar')
plt.show()


**Q7.** Repeat Q5 with `kind='violin'`, and add `row='Default'` on top of `col='Education_Level'` so you get a full 2x3 grid of violin plots (`y='Credit_Score'`, no `x=` needed since row/col already split the categories).

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
sns.catplot(data=df, y='Credit_Score', row='Default', col='Education_Level', kind='violin', height=3.5)
plt.show()


**Q8.** Repeat Q5's boxplot catplot, but pass `sharey=False` — each Education Level panel gets its own y-axis scale instead of a shared one. This matters when facets genuinely have very different scales and forcing a shared axis would flatten the smaller-scale panels.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box', sharey=False)
plt.show()


**Q9.** Both `FacetGrid` and `catplot` return a `FacetGrid`-type object with its own customization methods. Build `g = sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')`, then call `g.set_titles("{col_name}")` (simplify each panel's title to just the category name) and `g.set_axis_labels("Default Status", "Debt-to-Income (%)")` (relabel every panel's axes at once).

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
g = sns.catplot(data=df, x='Default', y='Debt_to_Income', col='Education_Level', kind='box')
g.set_titles("{col_name}")
g.set_axis_labels("Default Status", "Debt-to-Income (%)")
plt.show()


**Q10 (Capstone).** Build a polished small-multiples chart: `sns.catplot(data=df, x='Default', y='Credit_Score', col='Education_Level', kind='violin', palette={0: 'seagreen', 1: 'tomato'}, height=4.5)`. Then customize the returned grid with `g.set_titles("Education: {col_name}")`, `g.set_axis_labels("Default Status", "Credit Score")`, and add an overall title via `g.fig.suptitle("Credit Score by Default Status, Faceted by Education")` followed by `g.fig.subplots_adjust(top=0.85)` so the title doesn't overlap the panels.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
g = sns.catplot(data=df, x='Default', y='Credit_Score', col='Education_Level', kind='violin',
                 hue='Default', palette={0: 'seagreen', 1: 'tomato'}, legend=False, height=4.5)
g.set_titles("Education: {col_name}")
g.set_axis_labels("Default Status", "Credit Score")
g.fig.suptitle("Credit Score by Default Status, Faceted by Education")
g.fig.subplots_adjust(top=0.85)
plt.show()


> **Checkpoint — Section 11:** `FacetGrid` + `.map()` is the general-purpose tool — it
> can facet *any* plotting function. `catplot` is the shortcut specifically for
> categorical chart types (`kind='box'/'bar'/'violin'/...`), skipping the manual `.map()`
> step. Both return an object with `.set_titles()`, `.set_axis_labels()`, and `.fig` for
> further customization, exactly like `JointGrid` and `ClusterGrid` before it.


---
## Section 12: Pairplot

`sns.pairplot()` is the ultimate first-look tool: it automatically builds a full grid of
every numeric column plotted against every other, with each column's own distribution
on the diagonal. It's the fastest way to scan an entire dataset for relationships — but
it does redraw every pair, so it can get slow (and visually overwhelming) with many
columns or a very large number of rows.


**Q11.** Select just `['Age', 'Annual_Income', 'Credit_Limit', 'Credit_Score', 'Debt_to_Income']` from `df` and plot `sns.pairplot()` on that subset — restricting the columns up front is the standard way to keep pairplot fast and readable, rather than throwing the whole DataFrame at it.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
pairplot_cols = ['Age', 'Annual_Income', 'Credit_Limit', 'Credit_Score', 'Debt_to_Income']
sns.pairplot(df[pairplot_cols])
plt.show()


**Q12.** Repeat Q11, but call `sns.pairplot(df[pairplot_cols + ['Default']], hue='Default', palette={0: 'steelblue', 1: 'tomato'})` — now every scatterplot in the grid is color-coded by default status, so you can spot at a glance which pairwise relationships actually separate defaulters from non-defaulters.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
sns.pairplot(df[pairplot_cols + ['Default']], hue='Default', palette={0: 'steelblue', 1: 'tomato'})
plt.show()


**Q13.** Repeat Q12, but change `diag_kind='kde'` (smooth density curves on the diagonal) instead of the default histogram — often reads more cleanly once you have several hue groups on the diagonal.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
sns.pairplot(df[pairplot_cols + ['Default']], hue='Default',
             palette={0: 'steelblue', 1: 'tomato'}, diag_kind='kde')
plt.show()


**Q14.** Rather than subsetting the DataFrame beforehand, use pairplot's own `vars=` parameter directly: `sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income'], hue='Default', palette={0: 'steelblue', 1: 'tomato'})`. This is the more common pattern in practice — pass the full DataFrame, and let `vars=` pick the columns.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income'],
             hue='Default', palette={0: 'steelblue', 1: 'tomato'})
plt.show()


**Q15.** Repeat Q14, adding `kind='reg'` — every off-diagonal panel now shows a fitted regression line through the scatter, giving you an at-a-glance sense of the *direction and strength* of each pairwise relationship, not just the raw scatter. (Note: this is noticeably slower since it fits a regression per panel, per hue group — exactly the tradeoff the phase description warns about.)

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income'], kind='reg')
plt.show()


**Q16.** Repeat Q14's hue-colored pairplot, adding `corner=True` — this hides the upper-triangle panels, which mirror the lower triangle exactly (just with axes swapped), cutting the number of panels drawn roughly in half without losing any information.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income'],
             hue='Default', palette={0: 'steelblue', 1: 'tomato'}, corner=True)
plt.show()


**Q17.** `sns.pairplot()` returns a `PairGrid` object. Capture it as `g`, print `type(g)`, then add an overall title with `g.fig.suptitle("Pairwise Relationships: Key Risk Features")` followed by `g.fig.subplots_adjust(top=0.93)`.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
g = sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income'],
                  hue='Default', palette={0: 'steelblue', 1: 'tomato'})
print(type(g))
g.fig.suptitle("Pairwise Relationships: Key Risk Features")
g.fig.subplots_adjust(top=0.93)
plt.show()


**Q18 (Capstone).** Build the final version: `sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income', 'Credit_Limit'], hue='Default', palette={0: 'seagreen', 1: 'tomato'}, corner=True, diag_kind='kde', plot_kws={'alpha': 0.6})`. Capture it as `g`, relabel the legend with `g._legend.set_title('Default')`, and add a suptitle with proper spacing.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
g = sns.pairplot(df, vars=['Annual_Income', 'Credit_Score', 'Debt_to_Income', 'Credit_Limit'],
                  hue='Default', palette={0: 'seagreen', 1: 'tomato'}, corner=True,
                  diag_kind='kde', plot_kws={'alpha': 0.6})
g._legend.set_title('Default')
g.fig.suptitle("Key Risk Feature Relationships, by Default Status", y=1.02)
plt.show()


---
## Checkpoint: Seaborn Phase 6 Complete

You've covered:
- **FacetGrid / catplot** — `FacetGrid` + `.map()` as the general small-multiples engine,
  `catplot(kind=...)` as the categorical-specific shortcut, `row=`/`col=`/`col_wrap=` for
  layout, `hue=` for within-panel color, `sharey=False` for independent scales, and
  `.set_titles()`/`.set_axis_labels()`/`.fig` for finishing touches
- **Pairplot** — the fastest whole-dataset first look, with `vars=` to control scope and
  speed, `hue=` for group comparison across every panel, `diag_kind='kde'`, `kind='reg'`
  for fitted trend lines (at a real speed cost), and `corner=True` to cut redundant panels

Between these two, you now have both a *targeted* small-multiples tool (facet on exactly
the variable you care about) and a *broad* one (scan everything at once) — the natural
last exploratory step before committing to which features go into a model.

**Where to next:** this is a strong stopping point for exploratory visualization. From
here, the natural move is applying this full toolkit to your real credit card risk
dataset, or moving toward feature engineering and modeling itself.
